# Session 2 · Day 1 - A Routable Road Network from OpenStreetMap

A center finder, a coverage map, and a travel-time estimate all rest on one object: a road network
that a computer can search for paths. This notebook builds that object from **OpenStreetMap** for a
small slice of Metro Manila, small enough to download and route in a few minutes. The sibling
notebook `S2_D2_01_center_recommendation` then uses the same kind of network to rank assessment
centers by road distance.

The network is built with **osmnx**, the roads are examined as a table with **geopandas**, and the
shortest paths come from **networkx**.

### What this notebook covers
1. What OpenStreetMap stores: nodes, ways, relations, and tags
2. Download a small drivable network around one point in Metro Manila
3. Read the roads as a table of edges, one row per segment
4. Sort the roads into a hierarchy by `highway` class
5. Check how complete the tags are, because the data is volunteer made
6. Turn the table into a routable graph: add speeds, add travel times, keep the largest strongly connected component
7. Trace one route two ways, shortest by distance and fastest by time, and compare both to the straight line
8. Scale the idea up to a whole region without downloading the country

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - 🌐 **One cell reaches the internet**: the OpenStreetMap download. It takes one to a few minutes, so read ahead while it runs.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Write code between those lines only.
> - ✅ After an exercise, run the **check cell**. Fix the code until it prints a pass.
> - ❓ **Question cells** take a short typed answer, checked against a value computed from the data.
> - 🐞 **Spot-the-flaw cells** show tempting but wrong code. Read, explain, then compare to the fix.
> - 🎛️ **Play-around cells** are open to tweak: change the area or the route, re-run, compare.

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code cells point out each pandas idiom as it appears.

In [ ]:
# Run once per Colab session (skip if already installed locally).
!pip -q install osmnx networkx geopandas shapely folium matplotlib mapclassify

In [ ]:
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import folium
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
ox.settings.log_console = False
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print("osmnx", ox.__version__, "· networkx", nx.__version__, "· geopandas", gpd.__version__)

## What OpenStreetMap stores

**OpenStreetMap (OSM)** is a free, editable map of the world built by volunteers: local mappers,
humanitarian groups, government contributors, and companies. Every road, building, and school in it
was traced or surveyed by a person. That is its strength (free, open, and often detailed in Metro
Manila) and also its caveat: coverage and accuracy vary from place to place, measured below under
tag completeness.

Inside OSM there are only three kinds of elements:

| Element | What it is | Example |
|---|---|---|
| **node** | a single point with latitude and longitude | a street corner, a school's pin |
| **way** | an ordered list of nodes | a stretch of EDSA, the outline of a building |
| **relation** | a group of nodes and ways | a bus route, a city boundary |

Everything else, including road type, name, speed limit, and one-way status, is stored as **tags**,
which are `key=value` pairs attached to those elements, such as `highway=primary` or
`name=Commonwealth Avenue`. A routable road network keeps the ways tagged `highway=...` that a car
can legally use, and the nodes where they meet.

## A small drivable network

The whole-country OSM file holds more than 7,000 islands and expands to many gigabytes in memory.
A point and a radius keep the download small: `ox.graph.graph_from_point` asks the Overpass service
for only the drivable roads within a set distance of one coordinate. `ox.graph.graph_from_bbox`
does the same job for a rectangle, which is handy when the area of interest is not round.

A radius of a few kilometres stays under a couple of minutes. A whole region does not, which is the
reason the scaling section at the end switches to a pre-built file.

In [ ]:
AREA_NAME = "Quezon City around SEAMEO INNOTECH"
CENTER_LAT, CENTER_LON = 14.6601097, 121.0565753   # SEAMEO INNOTECH, Commonwealth Ave, QC
RADIUS_M = 4000                              # metres around the centre point

t0 = time.time()
G_raw = ox.graph.graph_from_point((CENTER_LAT, CENTER_LON), dist=RADIUS_M,
                                  network_type="drive", simplify=True)   # drive = car-legal roads; simplify folds node chains into one edge per segment
print(f"Downloaded a drive network for {AREA_NAME}: "
      f"{len(G_raw):,} nodes, {G_raw.number_of_edges():,} edges "
      f"(in {time.time() - t0:.0f} s)")

## Roads as a table of edges

A graph has two matching tables behind it. `ox.graph_to_gdfs` returns both: the **nodes**
(intersections and dead ends) and the **edges** (the road segments between them). Each edge row is
one segment of road from node `u` to node `v`. The OSM tags became columns, osmnx added `length` in
**metres**, and the `geometry` column holds the shape of the line.

Some tags hold a list when one way carries several values, so the first value is taken to get a
single road class per segment, with the `_link` suffix of on-ramps and off-ramps folded into the
parent class.

In [ ]:
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G_raw, nodes=True, edges=True)   # split the graph into two GeoDataFrames

def first_value(v):
    """Some OSM tags hold a list when one way carries several values. Keep the first."""
    return v[0] if isinstance(v, list) else v

edges_gdf["road_class"] = (edges_gdf["highway"].map(first_value)        # .map runs first_value down the column
                           .str.replace("_link", "", regex=False))      # fold on/off-ramps into the parent class

total_km = edges_gdf["length"].sum() / 1000
named = edges_gdf.loc[edges_gdf["name"].notna(), "name"].map(first_value).nunique()   # .loc[mask, col] filters rows then picks one column
print(f"{len(edges_gdf):,} road segments · {len(nodes_gdf):,} nodes · "
      f"{total_km:,.1f} km of drivable road · {named:,} distinct road names")

show_cols = [c for c in ["name", "highway", "oneway", "maxspeed", "lanes", "length", "geometry"]
             if c in edges_gdf.columns]   # keep only the columns that exist, since OSM tags vary by area
edges_gdf[show_cols].head(8)

## The road hierarchy

OSM classifies every road with the `highway` tag. In NCR the classes roughly map to:

| OSM class | Typical NCR examples |
|---|---|
| `motorway` | NLEX, SLEX, Skyway, NAIAX, CAVITEX |
| `trunk` | EDSA, C-5, Roxas Boulevard |
| `primary`, `secondary` | major city arterials |
| `tertiary`, `unclassified` | connectors between barangays |
| `residential`, `living_street` | streets inside neighbourhoods |
| `service` | access roads and parking aisles |

The tagging depends on the volunteers who mapped each area, so the examples are typical. The chart shows how many kilometres of each class exist, and the map shows where they
are.

In [ ]:
ORDER = ["motorway", "trunk", "primary", "secondary", "tertiary",
         "unclassified", "residential", "living_street", "service"]
COLORS = {"motorway": "#b2182b", "trunk": "#ef8a62", "primary": "#f4a582", "secondary": "#2166ac",
          "tertiary": "#67a9cf", "unclassified": "#999999", "residential": "#bbbbbb",
          "living_street": "#cccccc", "service": "#dddddd", "other": "#eeeeee"}
WIDTH = {"motorway": 2.2, "trunk": 1.8, "primary": 1.3, "secondary": 1.0, "tertiary": 0.7}

km = edges_gdf.groupby("road_class")["length"].sum() / 1000   # metres per class -> km (SQL GROUP BY + SUM)
km_ord = km.reindex(ORDER).dropna()                           # force the hierarchy order, drop classes not present
other = km[~km.index.isin(ORDER)].sum()                       # ~isin = NOT IN: everything outside the named classes
if other > 0:
    km_ord["other"] = other

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1.4]})
vals = km_ord.values[::-1]            # [::-1] reverses, so the longest class lands at the top of the bar chart
labs = list(km_ord.index[::-1])
a1.barh(labs, vals, color=[COLORS.get(c, "#dddddd") for c in labs])
for y, v in enumerate(vals):
    a1.text(v, y, f" {v:,.1f} km", va="center", fontsize=8)
a1.set_xlim(0, max(vals) * 1.22)
a1.set_title("Drivable road length by class")
a1.set_xlabel("km")

e3857 = edges_gdf.to_crs(3857)        # reproject to Web Mercator for drawing on a flat axis
for cls in ORDER[::-1]:
    sub = e3857[e3857["road_class"] == cls]
    if len(sub):
        sub.plot(ax=a2, color=COLORS[cls], linewidth=WIDTH.get(cls, 0.4), label=cls)
a2.set_axis_off()
a2.set_title(f"Drivable network around {AREA_NAME}")
a2.legend(loc="lower left", fontsize=7, frameon=True)
plt.tight_layout()
plt.show()

## How complete are the tags

Before using a column, check how often it is actually filled in. The table shows the share of road
**kilometres** that carry each tag, broken down by road class.

Look at `maxspeed`. Where it is missing, a travel-time estimate has to rely on an assumed speed,
and that assumption must be stated whenever the result is shared. The next section makes exactly
that assumption.

In [ ]:
TAGS = [t for t in ["name", "maxspeed", "lanes"] if t in edges_gdf.columns]
rows = {}
for cls in ORDER + ["ALL ROADS"]:
    sub = edges_gdf if cls == "ALL ROADS" else edges_gdf[edges_gdf["road_class"] == cls]
    if len(sub) == 0:
        continue
    w = sub["length"]
    rows[cls] = {t: (w * sub[t].notna()).sum() / w.sum() * 100 for t in TAGS}   # length-weighted share of km where the tag is filled
comp = pd.DataFrame(rows).T.round(0)   # .T transposes: road class becomes the row, tag the column
comp.index.name = "road class"
comp.style.format("{:.0f}%")

## From a table to a routable graph

A table of lines can be drawn, but it cannot answer "how to get from A to B". That needs a
**graph**, the core idea of network science:

- **Nodes** are intersections and dead ends.
- **Edges** are the road segments connecting them, and they are **directed**, because a one-way
  street runs only one way.
- Each edge carries **weights** such as length in metres or travel time in seconds.

Routing by distance is straightforward. Routing by time needs a speed on every edge. Where OSM has
a `maxspeed` tag osmnx uses it. Everywhere else a speed is **assumed** per road class. The values
below are rough free-flow assumptions for training only. They do not model Metro Manila traffic,
which is why a real trip takes longer.

One cleanup step matters. Some parts of the network cannot be reached from the rest, or cannot be
left once entered: a gated subdivision mapped without its gate, a one-way street tagged the wrong
way, or a road cut off by the download radius. Keeping only the **largest strongly connected
component**, the part where every node can reach every other node, removes those traps before any
route is searched.

In [ ]:
PH_SPEEDS = {   # km/h, used only where OSM has no maxspeed tag
    "motorway": 80, "motorway_link": 40, "trunk": 50, "trunk_link": 30,
    "primary": 40, "primary_link": 30, "secondary": 30, "secondary_link": 25,
    "tertiary": 30, "tertiary_link": 20, "unclassified": 20, "residential": 20,
    "living_street": 10, "service": 10,
}

G_core = ox.truncate.largest_component(G_raw, strongly=True)   # keep the mutually reachable core
G = ox.routing.add_edge_speeds(G_core, hwy_speeds=PH_SPEEDS)    # adds speed_kph to every edge
G = ox.routing.add_edge_travel_times(G)                        # adds travel_time (seconds)

kept = len(G) / len(G_raw)
print(f"Raw graph: {len(G_raw):,} nodes, {G_raw.number_of_edges():,} edges")
print(f"Connected core: {len(G):,} nodes ({kept:.1%} kept), {G.number_of_edges():,} edges")
print("Edge weights now available:", "length (m), speed_kph, travel_time (s)")

## A worked route between two points in Quezon City

Two coordinates in, a route out. The origin and destination are snapped to their nearest road
nodes, and a small helper sums any edge weight along a path (picking the lighter edge where two
nodes are joined more than once).

Two weights give two routes. The **shortest** route minimises metres. The
**fastest** route minimises the assumed seconds, so it prefers a longer run on a fast road over a
short crawl through side streets. Both are then compared to the **straight line**, the great-circle
distance that no trip can beat. The ratio of road distance to straight-line distance is the detour
factor, and it is rarely 1.

In [ ]:
ORIGIN_NAME = "SEAMEO INNOTECH"
ORIGIN_LAT, ORIGIN_LON = 14.6601097, 121.0565753
DEST_NAME = "Quezon Memorial Circle"
DEST_LAT, DEST_LON = 14.6515, 121.0493

orig = ox.distance.nearest_nodes(G, X=ORIGIN_LON, Y=ORIGIN_LAT)   # snap each point to a road node
dest = ox.distance.nearest_nodes(G, X=DEST_LON, Y=DEST_LAT)

def path_cost(G, route, attr):
    """Sum one edge attribute along a node path, using the lightest parallel edge."""
    total = 0.0
    for u, v in zip(route[:-1], route[1:]):                                   # walk the path node by node
        data = min(G[u][v].values(), key=lambda d: d.get(attr, float("inf")))   # G[u][v] holds every parallel edge; take the lightest
        total += data[attr]
    return total

print("Snapped origin to node", orig, "and destination to node", dest)

## ✏️ Exercise - the two routes

The graph `G` carries two edge weights: `length` in metres and `travel_time` in seconds. Use
`nx.shortest_path` to find two node paths from `orig` to `dest` and store them in the given
variables:

- **`route_short`**: the path that minimises **distance** (weight `"length"`).
- **`route_fast`**: the path that minimises **time** (weight `"travel_time"`).

Complete the body between the markers. *Hint:* `nx.shortest_path(G, orig, dest, weight="length")`
returns a list of nodes from `orig` to `dest`.

In [ ]:
### START CODE HERE ###   (about 2 lines)
route_short = None   # path from orig to dest, weight "length"
route_fast = None    # path from orig to dest, weight "travel_time"
### END CODE HERE ###
print("route_short:", None if route_short is None else f"{len(route_short)} nodes",
      "· route_fast:", None if route_fast is None else f"{len(route_fast)} nodes")

## ✅ Check the result
Run this. It measures both paths and confirms each minimises its own weight.

In [ ]:
try:
    assert route_short is not None and route_fast is not None, "Fill in both routes first."
    assert route_short[0] == orig and route_short[-1] == dest, "route_short must run orig to dest."
    assert route_fast[0] == orig and route_fast[-1] == dest, "route_fast must run orig to dest."
    short_m, fast_m = path_cost(G, route_short, "length"), path_cost(G, route_fast, "length")
    short_s, fast_s = path_cost(G, route_short, "travel_time"), path_cost(G, route_fast, "travel_time")
    assert short_m <= fast_m + 1e-6, "route_short should not be longer in metres than route_fast."
    assert fast_s <= short_s + 1e-6, "route_fast should not be slower than route_short."
    print(f"✅ Passed! Shortest = {short_m/1000:.2f} km / {short_s/60:.1f} min · "
          f"Fastest = {fast_m/1000:.2f} km / {fast_s/60:.1f} min.")
except AssertionError as e:
    print("❌ Not yet:", e)

The shortest route has the smaller distance and the fastest route has the smaller
time, and the two paths need not be the same roads. Both exceed the straight line: road distance in
Metro Manila is often 1.3 to 2 times the straight-line distance, and the factor changes from place
to place. That gap is the reason center recommendation and coverage questions rank by road
distance.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):   # great-circle distance in metres, the straight line no road can beat
    R = 6_371_000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

straight = haversine_m(ORIGIN_LAT, ORIGIN_LON, DEST_LAT, DEST_LON)
summary = pd.DataFrame({
    "distance (km)": [straight / 1000,
                      path_cost(G, route_short, "length") / 1000,
                      path_cost(G, route_fast, "length") / 1000],
    "time (min, no traffic)": [np.nan,
                               path_cost(G, route_short, "travel_time") / 60,
                               path_cost(G, route_fast, "travel_time") / 60],
    "road segments": [np.nan, len(route_short) - 1, len(route_fast) - 1],
}, index=["straight line", "shortest route", "fastest route"]).round(2)
ratio = path_cost(G, route_short, "length") / straight   # detour factor: road distance over straight line
print(f"Road distance is {ratio:.2f} times the straight-line distance.")
summary

A map makes the two routes concrete: the shortest in solid blue, the fastest dashed in red.

In [ ]:
m = folium.Map(location=[(ORIGIN_LAT + DEST_LAT) / 2, (ORIGIN_LON + DEST_LON) / 2],
               zoom_start=14, tiles="CartoDB positron")
pts_s = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in route_short]   # node id -> (lat, lon); osmnx stores y=lat, x=lon
pts_f = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in route_fast]
folium.PolyLine(pts_s, color="#2166ac", weight=6, opacity=0.7, tooltip="shortest (distance)").add_to(m)
folium.PolyLine(pts_f, color="#d1495b", weight=4, opacity=0.9, dash_array="6",
                tooltip="fastest (time)").add_to(m)
folium.Marker([ORIGIN_LAT, ORIGIN_LON], tooltip=ORIGIN_NAME,
              icon=folium.Icon(color="green")).add_to(m)
folium.Marker([DEST_LAT, DEST_LON], tooltip=DEST_NAME,
              icon=folium.Icon(color="red")).add_to(m)
m

### 🐞 Spot the flaw

A teammate routes without naming a weight, arguing the result is "the shortest path anyway":

```python
route_hops = nx.shortest_path(G, orig, dest)
```

It runs and returns a path. Read the next cell, which measures that path in metres against the
shortest-by-distance route, then explain in one sentence what `nx.shortest_path` minimises when no
weight is given.

In [ ]:
route_hops = nx.shortest_path(G, orig, dest)   # no weight: minimises the number of hops, not distance
hops_km = path_cost(G, route_hops, "length") / 1000
best_km = path_cost(G, route_short, "length") / 1000
print(f"No-weight path: {len(route_hops) - 1} segments, {hops_km:.2f} km")
print(f"Shortest-by-distance path: {len(route_short) - 1} segments, {best_km:.2f} km")
print(f"The no-weight path is {hops_km / best_km:.2f} times as long on the ground.")

**The flaw:** with no weight, `nx.shortest_path` minimises the number of segments (hops). It does not minimise distance or time. It treats a two kilometre stretch of EDSA and a twenty metre alley as equal
steps, so the fewest-segments path can be far longer on the ground. Routing needs an explicit
`weight`, either `length` for distance or `travel_time` for time, which is what the two routes
above use.

### ❓ Question

How many road segments (edges) does the largest strongly connected component hold? The connected
core graph is `G`. Fill in the integer count.

In [ ]:
q_answer = None   # <- an integer count of edges in G

In [ ]:
ref = G.number_of_edges()
try:
    assert q_answer is not None, "q_answer is still None. Enter the count."
    assert int(q_answer) == ref, f"expected {ref}, got {q_answer}"
    print(f"✅ Passed! The connected core holds {ref:,} edges.")
except AssertionError as e:
    print("❌ Not yet:", e)

## 🎛️ Play around - a different area or route

Two choices drive this notebook: the **area** downloaded at the top and the **two points** routed
across it. Change `CENTER_LAT`, `CENTER_LON`, and `RADIUS_M` to move or resize the area, then re-run
from **A small drivable network**. Keep the radius under about 5 km so the download stays quick. A
larger radius pulls in more road classes and a longer build. Move `DEST_LAT` and `DEST_LON` to trace
a route that runs across Quezon City, where the detour factor climbs because the crossings are few.
The cell below prints the current choices and the road classes present.

In [ ]:
print("Current area:", AREA_NAME, f"({CENTER_LAT}, {CENTER_LON}), radius {RADIUS_M} m")
print("Road classes present:", ", ".join(sorted(edges_gdf["road_class"].dropna().unique())))
print("A longer route to try: destination near (14.6400, 121.0430).")

## Scale up without downloading the country

One small network around one point is cheap. A regional, always-on service needs the same idea at a
larger size, without a live download on every request.

- **Pre-build the network from a file, do not download per request.** Geofabrik publishes the whole
  Philippines as a daily `.osm.pbf`. A clipped copy for Metro Manila, `ncr_clip.osm.pbf`, sits on
  the public bucket at
  `https://storage.googleapis.com/tesda-datasets/data/ncr_clip.osm.pbf`. A region-wide drive graph
  built once from that file and saved loads in seconds, so the live Overpass download above is a
  teaching convenience, and the pre-built graph is what production uses.
- **Use a dedicated engine at national scale.** For many origins and destinations the standard
  engine is **OSRM**, which answers one origin-to-destination query per request after a one-time
  preprocessing step. The notebook `S2_D2_00_osrm_routing` sets that up.

The cap of a few kilometres is deliberate. Building or saving a full regional graph
produces a file of hundreds of megabytes, which belongs behind the pre-build step above. A teaching run should not carry it.

## Recap

- OpenStreetMap stores the map as **nodes**, **ways**, and **relations**, with everything else
  carried as **tags** such as `highway=primary`.
- The same roads read two ways: a **table of edges** for counting, summarising, and checking tag
  completeness, and a **directed graph** for routing.
- A routable graph needs **speeds** and **travel times** on every edge, and a **largest strongly
  connected component** so every node can reach every other.
- One route traced two ways shows that **shortest by distance** and **fastest by time** differ, and
  both exceed the **straight line** by a detour factor that is rarely 1.
- A regional service keeps this cheap with a **pre-built network** or **OSRM**, avoiding a live download per request.

**Next:** `S2_D2_01_center_recommendation` ranks assessment centers for a learner on exactly this
kind of road network.

## References & further reading

Documentation and background for the tools and methods in this notebook.

- [OpenStreetMap, about](https://www.openstreetmap.org/about)
- [OpenStreetMap wiki, the highway tag](https://wiki.openstreetmap.org/wiki/Key:highway)
- [OSMnx, documentation](https://osmnx.readthedocs.io/)
- [OSMnx, get a graph from a point](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.graph.graph_from_point)
- [OSMnx, speeds and travel times](https://osmnx.readthedocs.io/en/stable/user-reference.html#module-osmnx.routing)
- [NetworkX, shortest paths (Dijkstra)](https://networkx.org/documentation/stable/reference/algorithms/shortest_paths.html)
- [Geofabrik, Philippines OSM downloads](https://download.geofabrik.de/asia/philippines.html)
- [Project OSRM (national-scale routing)](https://project-osrm.org/)
- [Haversine formula (overview)](https://en.wikipedia.org/wiki/Haversine_formula)